# 🕷️ Crawl Only - VifactCheck Dataset

**Extract articles from VifactCheck dataset using CrawlerFactory**

📤 Output: `./src/data/json/news_data_vifactcheck_*.json`

In [ ]:
# Install dependencies
import subprocess
import sys

packages = ['loguru', 'tqdm', 'beautifulsoup4', 'lxml', 'httpx', 'nest-asyncio', 'datasets']

for pkg in packages:
    try:
        __import__(pkg.replace('-', '_'))
        print(f"✅ {pkg}")
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])
        print(f"📦 Installed {pkg}")

In [ ]:
# Setup
import sys
import os
import nest_asyncio
from pathlib import Path

nest_asyncio.apply()
os.environ["OPENSSL_CONF"] = "openssl.cnf"

project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'src'))

print(f"✅ Ready: {project_root}")

In [ ]:
# Import crawler modules
try:
    from src.crawler.crawler_factory import CrawlerFactory
    from src.processing.dataset_handler import DatasetHandler
    print("✅ Crawler modules imported")
except ImportError as e:
    print(f"❌ Import error: {e}")

In [ ]:
# Configuration
CRAWLING_CONFIG = {
    "dataset_name": "tranthaihoa/vifactcheck",
    "url_column": "Url",
    "splits": ["train", "test", "dev"],
    "url_limit": None,  # ALL URLs
    "cache_dir": "./data/caches",
    "output_dir": "./src/data/json"
}

os.makedirs(CRAWLING_CONFIG["cache_dir"], exist_ok=True)
os.makedirs(CRAWLING_CONFIG["output_dir"], exist_ok=True)

print("🔧 Configuration:")
for k, v in CRAWLING_CONFIG.items():
    print(f"  • {k}: {v}")

In [ ]:
async def crawl_dataset():
    """Crawl VifactCheck dataset"""
    results = []
    
    for split in CRAWLING_CONFIG["splits"]:
        print(f"\n--- Processing {split} ---")
        
        crawler_factory = CrawlerFactory(
            cache_filename=f"{CRAWLING_CONFIG['cache_dir']}/crawling_status_{split}.json",
            failed_log_filename=f"{CRAWLING_CONFIG['cache_dir']}/failed_urls_{split}.json"
        )
        
        if crawler_factory.check_cache_file_exists():
            crawler_factory.clear_cache()
        
        dataset_handler = DatasetHandler(CRAWLING_CONFIG["dataset_name"])
        urls = dataset_handler.get_urls_from_split(
            split=split, 
            url_column=CRAWLING_CONFIG["url_column"], 
            limit=CRAWLING_CONFIG["url_limit"]
        )
        
        if urls:
            print(f"Found {len(urls)} URLs")
            output_filename = f"news_data_vifactcheck_{split}.json"
            await crawler_factory.crawl_and_save_all(urls, output_filename, format_name="custom")
            results.append({"split": split, "count": len(urls)})
            print(f"✅ {split} completed")
    
    return results

print("🔧 Crawl function ready")

In [ ]:
# Start crawling
results = await crawl_dataset()
print(f"\n✅ Crawled {len(results)} splits")
for r in results:
    print(f"  • {r['split']}: {r['count']} URLs")

In [ ]:
# Check results
import json
from pathlib import Path

print("🚀 Crawling Results:")
data_dir = Path(CRAWLING_CONFIG["output_dir"])

for split in CRAWLING_CONFIG["splits"]:
    json_file = data_dir / f"news_data_vifactcheck_{split}.json"
    if json_file.exists():
        with open(json_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f"📁 {split}: {len(data)} articles")
    else:
        print(f"❌ {split}: Not found")